# Sprint 3: Gönderi Kümeleme ve Profil Analizi

Veri setinde tek bir ajan (author_id) bulunduğu için, bu sprint kapsamında analizler **gönderi bazlı** olarak gerçekleştirilmiştir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

## 1. Veri Yükleme ve Özellik Mühendisliği (7 Özellik)

Her gönderi için profil oluşturacak 7 sayısal özellik türetiyoruz.

In [ ]:
df = pd.read_csv('moltbook_final_v4.csv')

# 1. toxic_level
# 2. upvotes_count

# 3. topic_encoded
le = LabelEncoder()
df['topic_encoded'] = le.fit_transform(df['topic_label'].astype(str))

# 4. text_len
df['text_len'] = df['content_body'].fillna('').str.len()

# 5. word_count
df['word_count'] = df['content_body'].fillna('').str.split().str.len()

# 6. special_char_count
df['special_char_count'] = df['content_body'].fillna('').apply(lambda x: len(re.findall(r'[!\?@#$]', x)))

# 7. is_code
df['is_code'] = df['content_body'].fillna('').str.contains('```').astype(int)

features = ['toxic_level', 'upvotes_count', 'topic_encoded', 'text_len', 'word_count', 'special_char_count', 'is_code']
post_profiles = df[['id'] + features].copy()

print(f"Gönderi profil tablosu boyutu: {post_profiles.shape}")
post_profiles.head()

## 2. Normalizasyon ve Elbow Yöntemi

In [ ]:
X = post_profiles[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Yöntemi (Optimal k=4 için Kontrol)')
plt.xlabel('Küme Sayısı')
plt.ylabel('WCSS')
plt.show()

## 3. K-Means Kümeleme (k=4)

In [ ]:
kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42)
post_profiles['cluster'] = kmeans.fit_predict(X_scaled)

print("Küme dağılımı:")
print(post_profiles['cluster'].value_counts())

## 4. PCA ile Görselleştirme

In [ ]:
pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
post_profiles['pca1'] = pca_results[:, 0]
post_profiles['pca2'] = pca_results[:, 1]

# Küme isimlerini tanımlayalım
cluster_names = {
    0: "Standart (Düşük Etkileşim)",
    1: "Teknik (Kod/Sembol)",
    2: "Viral (Yüksek Toksisite/Beğeni)",
    3: "Kaliteli (Toksik Değil/Yüksek Beğeni)"
}
post_profiles['cluster_name'] = post_profiles['cluster'].map(cluster_names)

plt.figure(figsize=(14, 9))
sns.scatterplot(data=post_profiles, x='pca1', y='pca2', hue='cluster_name', palette='Set1', alpha=0.7, s=100)
plt.title('Gönderi Davranış Kümeleri (PCA Analizi)', fontsize=16, pad=20)
plt.xlabel('Ana Bileşen 1 (Varyansın Çoğunu Açıklar)', fontsize=12)
plt.ylabel('Ana Bileşen 2 (İkincil Varyansı Açıklar)', fontsize=12)
plt.legend(title='İçerik Kümeleri', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=150)
plt.show()

## 5. Küme Profilleri ve Kayıt

In [ ]:
cluster_summary = post_profiles.groupby('cluster')[features].mean()
print("Küme Özellik Ortalamaları:")
display(cluster_summary)

post_profiles.to_csv('gonderi_kumeleri.csv', index=False)
print("\n✅ 'gonderi_kumeleri.csv' ve 'pca_scatter.png' başarıyla oluşturuldu.")